# Notebook 03 — Why Is My Customer Lookup So Slow?
## Diagnosing and Fixing Account-Level Query Performance

**What you'll learn**: Your date-range reports are fast (Notebook 02), but looking up a specific customer takes forever. Here's why — and what to ask your platform team to fix.

| | After Fix (Clustered) | Before Fix (Full Scan) |
|---|---|---|
| **Customer lookup** | <1% of data scanned | Scans the ENTIRE table |
| **Speed** | Sub-second | Seconds to minutes |
| **Action needed** | Ask platform team for a cluster key | None — this is the default |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

ALTER SESSION SET USE_CACHED_RESULT = FALSE;

---
## Step 1: The Problem — Why Does a Single Customer Lookup Read the Entire Table?

You need to pull all transactions for account `ACC-0000012345`. Sounds simple, right?  
But the table has 500M rows loaded in date order — **your customer's transactions are scattered across the entire table**, mixed in with everyone else's transactions from the same dates.

In [ ]:
-- WORST CASE: account_id lookup on chronologically-loaded table (full scan)
SELECT COUNT(*) AS txn_count, MIN(transaction_date) AS first_txn, MAX(transaction_date) AS last_txn
FROM TRANSACTIONS
WHERE account_id = 'ACC-0000012345';

**Check Query Profile** → Notice: Snowflake scanned the ENTIRE table (~4 GB) just to find one customer's ~5,000 transactions!

---
## Step 2: The Fix — Your Platform Team Reorganizes the Table by `account_id`

When your platform team adds a **cluster key** on `account_id`, Snowflake physically reorganizes the data so each customer's transactions are stored together. This lets Snowflake jump directly to the right spot instead of reading everything.

*(Below simulates what Automatic Clustering achieves — in production, you just request the cluster key and Snowflake does this in the background.)*

In [ ]:
-- Use XL for creating the 500M-row clustered copy
USE WAREHOUSE HOL_XL;

-- Create clustered copy sorted by account_id (select key columns for efficient pruning)
CREATE OR REPLACE TABLE TRANSACTIONS_CLUSTERED AS
SELECT
    transaction_id,
    account_id,
    transaction_date,
    merchant_name,
    amount,
    channel
FROM TRANSACTIONS
ORDER BY account_id;

In [ ]:
-- Switch back to XS for query testing
USE WAREHOUSE HOL_XS;

---
## Step 3: After the Fix — Same Query, Dramatically Faster

In [ ]:
-- BEST CASE: Same query on clustered table (99%+ pruning)
SELECT COUNT(*) AS txn_count, MIN(transaction_date) AS first_txn, MAX(transaction_date) AS last_txn
FROM TRANSACTIONS_CLUSTERED
WHERE account_id = 'ACC-0000012345';

**Check Query Profile** → Compare:
- **Before** (no clustering): ALL data scanned, ~4 GB read
- **After** (clustered on account_id): Only relevant partitions scanned, ~0.01 GB read
- **Result: 500x less data scanned!**

---
## Step 4: Side-by-Side Metrics

In [ ]:
-- Compare performance
SELECT
    CASE WHEN query_text ILIKE '%TRANSACTIONS_CLUSTERED%' THEN 'CLUSTERED'
         ELSE 'UNCLUSTERED (full scan)' END AS version,
    total_elapsed_time / 1000 AS elapsed_sec,
    bytes_scanned / (1024*1024*1024) AS gb_scanned
FROM TABLE(SNOWFLAKE.INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(minute, -30, CURRENT_TIMESTAMP()),
    END_TIME_RANGE_END => CURRENT_TIMESTAMP(),
    RESULT_LIMIT => 50
))
WHERE query_text ILIKE '%ACC-0000012345%'
  AND query_type = 'SELECT'
  AND query_text NOT ILIKE '%QUERY_HISTORY%'
ORDER BY start_time DESC
LIMIT 2;

---
## Step 6: When NOT to Request Clustering (Anti-Pattern)

In [ ]:
-- currency_code has only 3 values — already naturally well-clustered
SELECT SYSTEM$CLUSTERING_INFORMATION('TRANSACTIONS', '(currency_code)');

---
## Key Takeaways

| | Good Request | Bad Request |
|---|---|---|
| **Cluster on** | The column you filter on most (`account_id`, `customer_id`) | A column with few values (`currency_code`, `channel`) |
| **Result** | 99%+ less data scanned, seconds instead of minutes | Zero improvement, wastes money on maintenance |
| **When to ask** | Your customer/account-level queries are slow | Your date-range reports are already fast |

**💡 What to tell your platform team**:  
*"My customer-level queries on TRANSACTIONS scan the entire table. Can we add a cluster key on `account_id`? Here's the Query Profile showing the full scan."*

**How to verify the fix worked**:  
Re-run your query and check the Query Profile — data scanned should drop by 99%+.

**Next** → [Notebook 04 — My Report Runs Out of Memory](./Notebook_04_Warehouse_Optimization.ipynb)